# Homework 13: iLQG and Reach Distance

**Computational Sensorimotor Control** | Week 13 of 15


## Setup

In [ ]:
# Install the smc package (run once)
!pip install --break-system-packages -q git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from smc import (
    Arm, Q_REF, make_hill_muscles,
    simulate_hill, lambda_for_posture, make_ramp,
    NX, NU, MUSCLE_NAMES, DT_SIM, DT_CTRL, N_SUBSTEPS,
    make_x0, make_target, set_muscle_state, hill_step, forward_rollout,
)

arm = Arm()
start_hand = arm.forward_kinematics(Q_REF)
T = 0.5
N = int(T / DT_CTRL)

# --- Paste your working functions from Lab 13 ---
# compute_jacobians, riccati_backward_ilqg, forward_pass, compute_cost, run_ilqg
# (These are provided in the starter notebook; paste your completed versions here.)

def compute_jacobians(x_bar, u_bar, eps=1e-5):
    A = np.zeros((NX, NX))
    B = np.zeros((NX, NU))
    for j in range(NX):
        xp = x_bar.copy(); xp[j] += eps
        xm = x_bar.copy(); xm[j] -= eps
        mp = make_hill_muscles(); set_muscle_state(mp, xp)
        mm = make_hill_muscles(); set_muscle_state(mm, xm)
        A[:, j] = (hill_step(xp, u_bar, mp) - hill_step(xm, u_bar, mm)) / (2 * eps)
    for j in range(NU):
        up = u_bar.copy(); up[j] += eps
        um = u_bar.copy(); um[j] -= eps
        mp = make_hill_muscles(); set_muscle_state(mp, x_bar)
        mm = make_hill_muscles(); set_muscle_state(mm, x_bar)
        B[:, j] = (hill_step(x_bar, up, mp) - hill_step(x_bar, um, mm)) / (2 * eps)
    return A, B

def riccati_backward_ilqg(A_list, B_list, Q, QN, R, x_nom, u_nom, xs):
    N = len(A_list)
    S = [None]*(N+1); v = [None]*(N+1)
    L = [None]*N; Lv = [None]*N; Lu = [None]*N
    S[N] = QN.copy()
    v[N] = QN @ (x_nom[N] - xs)
    for t in range(N-1, -1, -1):
        A, B = A_list[t], B_list[t]
        H_inv = np.linalg.inv(B.T @ S[t+1] @ B + R)
        L[t] = H_inv @ B.T @ S[t+1] @ A
        Lv[t] = H_inv @ B.T
        Lu[t] = H_inv @ R
        A_cl = A - B @ L[t]
        S[t] = A.T @ S[t+1] @ A_cl + Q
        v[t] = A_cl.T @ v[t+1] - L[t].T @ R @ u_nom[t] + Q @ x_nom[t]
    return L, Lv, Lu, S, v

def forward_pass(x0, u_nom, x_nom, L, Lv, Lu, v, alpha=1.0):
    N = len(u_nom)
    x_new = np.zeros((N+1, NX)); u_new = np.zeros((N, NU))
    x_new[0] = x0
    muscles = make_hill_muscles(); set_muscle_state(muscles, x0)
    for t in range(N):
        dx = x_new[t] - x_nom[t]
        du = -L[t] @ dx - alpha * Lv[t] @ v[t+1] - alpha * Lu[t] @ u_nom[t]
        u_new[t] = np.clip(u_nom[t] + du, 0.0, 1.0)
        x_new[t+1] = hill_step(x_new[t], u_new[t], muscles)
    return x_new, u_new

def compute_cost(x_traj, u_seq, QN, R, xs):
    N = len(u_seq)
    dx_N = x_traj[N] - xs
    cost = 0.5 * dx_N @ QN @ dx_N
    for t in range(N): cost += 0.5 * u_seq[t] @ R @ u_seq[t]
    return cost

def run_ilqg(x0, xs, u_init, QN, Q, R, max_iter=8, tol=1e-6):
    N = len(u_init)
    u_curr = u_init.copy()
    x_curr = forward_rollout(x0, u_curr)
    cost_curr = compute_cost(x_curr, u_curr, QN, R, xs)
    costs = [cost_curr]
    for it in range(max_iter):
        A_list, B_list = [], []
        for t in range(N):
            A, B = compute_jacobians(x_curr[t], u_curr[t])
            A_list.append(A); B_list.append(B)
        L, Lv, Lu, S, v = riccati_backward_ilqg(A_list, B_list, Q, QN, R, x_curr, u_curr, xs)
        improved = False
        for alpha in [1.0, 0.5, 0.25, 0.1, 0.05, 0.01]:
            x_try, u_try = forward_pass(x0, u_curr, x_curr, L, Lv, Lu, v, alpha)
            cost_try = compute_cost(x_try, u_try, QN, R, xs)
            if cost_try < cost_curr:
                x_curr = x_try.copy(); u_curr = u_try.copy(); cost_curr = cost_try
                improved = True; break
        costs.append(cost_curr)
        if not improved or (len(costs) >= 2 and abs(costs[-2] - costs[-1]) < tol):
            break
    return x_curr, u_curr, costs

# Cost matrices
QN = np.zeros((NX, NX))
QN[:4, :4] = np.diag([5000., 5000., 100., 100.])
Q = np.zeros((NX, NX))
R = 0.001 * np.eye(NU)

print("Functions loaded.")


## Part 1: Run iLQG for Four Distances (1 line)

Run iLQG for 8, 12, 16, and 20 cm reaches. The loop handles target setup, EPH init, and calling run_ilqg. You fill in one line to set the target hand position.


In [ ]:
distances = [0.08, 0.12, 0.16, 0.20]  # metres
results = {}

for dist in distances:
    print(f"\n{'='*50}")
    print(f"Distance: {dist*100:.0f} cm")
    
    ### YOUR CODE: set target hand position ###
    # target = ...
    
    q_tgt = arm.inverse_kinematics(*target)
    x0 = make_x0(Q_REF)
    xs = make_target(q_tgt)
    
    # EPH initial guess
    li = lambda_for_posture(Q_REF)
    lf = lambda_for_posture(q_tgt)
    lam_fn = make_ramp(li, lf, 0.05, 0.25)
    _, _, _, stim_eph = simulate_hill(lam_fn, T=T, dt=DT_SIM)
    u_init = np.zeros((N, NU))
    for t in range(N):
        u_init[t] = np.mean(stim_eph[t*N_SUBSTEPS:(t+1)*N_SUBSTEPS], axis=0)
    
    # Run iLQG
    x_conv, u_conv, costs = run_ilqg(x0, xs, u_init, QN, Q, R)
    
    h = arm.forward_kinematics(x_conv[-1, :2])
    err = np.linalg.norm(np.array(h) - target) * 100
    n_iter = len(costs) - 1
    print(f"  Converged in {n_iter} iterations, endpoint error: {err:.3f} cm")
    
    results[dist] = {
        'costs': costs, 'x_conv': x_conv, 'u_conv': u_conv,
        'target': target, 'xs': xs, 'x0': x0, 'err': err, 'n_iter': n_iter
    }

# Q1 answer
print("\n--- Q1 Answer ---")
for dist in distances:
    r = results[dist]
    print(f"  {dist*100:.0f} cm: {r['n_iter']} iterations, {r['err']:.3f} cm error")


**Q1 Answer: YOUR ANSWER HERE**


## Part 2: Convergence Curves (1 line)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#2E86AB', '#27AE60', '#F39C12', '#E74C3C']

for dist, color in zip(distances, colors):
    costs = results[dist]['costs']
    ### YOUR CODE: plot the convergence curve ###
    # ax.semilogy(...)

ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Cost (log scale)', fontsize=12)
ax.set_title('Part 2: Convergence curves — Hill-type muscle plant\n(cf. lecture Figure 4E, which was torque-driven)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


**Q2 Answer: YOUR ANSWER HERE**


## Part 3: B_t Variation Across Distances (2 lines)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
from scipy.ndimage import uniform_filter1d

bt_ratios = []

for dist, color in zip(distances, colors):
    x_conv = results[dist]['x_conv']
    u_conv = results[dist]['u_conv']
    
    # Compute Jacobians along converged trajectory
    pec_effects = []
    for t in range(N):
        _, B = compute_jacobians(x_conv[t], u_conv[t])
        ### YOUR CODE: compute pec mechanical effect (norm of B[:4, 0]) ###
        # pec_effect = ...
        pec_effects.append(pec_effect)
    
    pec_effects = np.array(pec_effects)
    pec_smooth = uniform_filter1d(pec_effects, size=15)
    
    ### YOUR CODE: compute variation ratio (max / min) ###
    # ratio = ...
    bt_ratios.append(ratio)
    
    t_ms = np.arange(N) * DT_CTRL * 1000
    axes[0].plot(t_ms, pec_smooth, color=color, lw=2,
                 label=f'{dist*100:.0f} cm (ratio: {ratio:.1f}×)')

axes[0].set_xlabel('Time (ms)', fontsize=11)
axes[0].set_ylabel('||B_t pec column|| (joint accel. effect)', fontsize=11)
axes[0].set_title('Part 3: Pec B_t variation across distances\n(extends lecture Figure 5A)',
                   fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Panel B: variation ratio vs distance
axes[1].plot([d*100 for d in distances], bt_ratios, 'ko-', lw=2, markersize=10)
axes[1].set_xlabel('Reach distance (cm)', fontsize=11)
axes[1].set_ylabel('B_t variation ratio (max/min)', fontsize=11)
axes[1].set_title('Pec B_t variation increases\nwith reach distance', fontsize=11, fontweight='bold')
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

for dist, ratio in zip(distances, bt_ratios):
    print(f"  {dist*100:.0f} cm: B_t ratio = {ratio:.1f}×")


**Q3 Answer: YOUR ANSWER HERE**


## Part 4: Summary Plot (1 line)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Panel A: iterations vs distance
### YOUR CODE: extract iteration count for each distance ###
# iters = ...

axes[0].plot([d*100 for d in distances], iters, 'ko-', lw=2.5, markersize=12)
axes[0].set_xlabel('Reach distance (cm)', fontsize=12)
axes[0].set_ylabel('Iterations to converge', fontsize=12)
axes[0].set_title('A) More nonlinearity →\n    more iterations', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Panel B: B_t ratio vs distance (from Part 3)
axes[1].plot([d*100 for d in distances], bt_ratios, 'ko-', lw=2.5, markersize=12)
axes[1].set_xlabel('Reach distance (cm)', fontsize=12)
axes[1].set_ylabel('B_t variation ratio (max/min)', fontsize=12)
axes[1].set_title('B) Longer reach →\n    more B_t variation', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

fig.suptitle('Part 4: Summary — reach distance determines linearization difficulty',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


**Q4 Answer: YOUR ANSWER HERE**
